In [28]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState

In [29]:
westeros_lore_chunks = [
    "House Lannister of Casterly Rock is the wealthiest of the Great Houses, their fortune built upon the deep gold mines of the Westerlands. Their unofficial motto is 'A Lannister always pays his debts.'",
    "House Tyrell of Highgarden commands the Reach, the most fertile region in Westeros. They provide the majority of the grain, fruit, and wine that sustains King's Landing, making their alliance crucial for the Crown's survival.",
    "Valyrian steel is an exceptionally sharp and durable metal forged with dragonfire and ancient magic in the Valyrian Freehold. It is one of the few known substances capable of instantly shattering White Walkers.",
    "The Alchemists' Guild in King's Landing possesses the secret to creating Wildfire, a highly volatile green liquid. Once ignited, Wildfire cannot be extinguished by water and burns hot enough to melt wood, stone, and even steel.",
    "The Iron Bank of Braavos is the wealthiest and most powerful financial institution in the known world. When princes or kings default on their loans, the Iron Bank is known to fund their enemies to ensure the debt is repaid.",
    "The Faceless Men are a guild of assassins based in the House of Black and White in Braavos. They worship the Many-Faced God of death and have the magical ability to change their appearances using the faces of the deceased.",
    "Dragonglass, known to the maesters as obsidian, is volcanic glass that can be found in abundance on the island of Dragonstone. Alongside Valyrian steel, it is the only other known weapon that can effectively kill White Walkers and their wights."
]

In [30]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
chat = ChatOllama(model="llama3", temperature=0.9)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1979.51it/s]


In [31]:
vector_store = FAISS.from_texts(westeros_lore_chunks, embeddings)


In [32]:
def save_secret_node(state: MessagesState):
    # Extract the text from the latest HumanMessage
    latest_message = state["messages"][-1].content
    
    # Optional: Clean the trigger word out before saving
    secret_text = latest_message.replace("Secret:", "").replace("Remember:", "").strip()
    
    # Write to LTM
    vector_store.add_texts([secret_text])
    
    # Return an AI message acknowledging the save without calling the LLM
    return {"messages": [AIMessage(content="Thank you for sharing this information. I have added it to my whispers.")]}
def chat_node(state: MessagesState):
    query = state["messages"][-1].content
    
    # Retrieve from LTM
    relevant_chunks = vector_store.similarity_search(query, k=3)
    context = "\n".join([chunk.page_content for chunk in relevant_chunks])
    
    # Build proper SystemMessage
    system_prompt = SystemMessage(
        content=f"You are Varys, the Master of Whisperers. Use this information to answer:\n\n{context}"
    )
    
    # Combine SystemMessage with the conversation history
    prompt_to_give = [system_prompt] + state["messages"]
    
    # Invoke and return
    response = chat.invoke(prompt_to_give)
    return {"messages": [response]}

def router(state: MessagesState):
    # Look directly at the string content of the user's message
    latest_text = state["messages"][-1].content.lower()
    
    if latest_text.startswith("secret:") or latest_text.startswith("remember:"):
        return "save_secret"
    else:
        return "chat"

In [33]:
builder = StateGraph(MessagesState)
checkpointer = InMemorySaver()

builder.add_node("save_secret", save_secret_node)
builder.add_node("chat", chat_node)

builder.add_conditional_edges(START, router, {
    "save_secret": "save_secret",
    "chat": "chat"
})

builder.add_edge("save_secret", END)
builder.add_edge("chat", END)

graph = builder.compile(checkpointer=checkpointer)